# The density route

Notebooks 01 and 02 modelled the **hazard** $h(t\mid x)$ — the instantaneous event
rate — and read survival off it as $S(t)=\exp\!\big(-\int_0^t h\big)$. That is the
*intensity route*: model a rate, then integrate.

There is a second, equally valid route. Instead of the rate, model the
**event-time density** $p(t\mid x)$ directly and read survival off it as

$$S(t) = 1 - \mathrm{CDF}(t), \qquad \mathrm{CDF}(t) = P(T < t \mid x).$$

TorchPP supports both. The intensity route subclasses `IntensityBasedTPP`
(notebooks 01–02); the density route subclasses `DensityBasedTPP`, which we use
here. Both families share one contract — `likelihood`, `survival_time`, `sample`,
`time_mean` — so a density model drops into the same fit-and-evaluate code and
compares head-to-head with an intensity model.

**Why a density survival model needs no encoder.** A neural density TPP (the
intensity-free model of Shchur et al., 2020) predicts its density parameters from
a recurrent embedding of the event history. In survival analysis each subject has
at most one event, so there is no history to encode: the density parameters are a
plain function of the static covariates $x$. That makes a concrete
`DensityBasedTPP` short — no encoder, just a map from $x$ to a distribution over
the event time.

In [ ]:
import math

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from torchpp import BatchedMVEventData, DensityBasedTPP, IntensityBasedTPP, TokenRoles
from torchpp.modules.nn.lognormal_mixture import LogNormalMixtureDistribution
from survival_tools import (
    make_weibull_ph,
    survival_batch, covariate_batch, fit_full_batch,
    model_survival_curve,
    kaplan_meier,
    plot_survival_km_vs_model,
)

# survival_tools selects the headless Agg backend on import; switch to inline
# afterwards so figures render in the notebook.
%matplotlib inline
_ = torch.manual_seed(0)

## 1. Single-event data with a rising hazard

We reuse the series' data generators. `make_weibull_ph` draws a **Weibull** event
time per subject: the baseline hazard $h_0(t)$ *rises* with time (shape $k=2$),
unlike the constant hazard of notebook 01. A constant-hazard model is therefore
mis-specified here — a good test for a flexible density model.

Each subject carries covariates $x$ and follows proportional hazards
$h(t\mid x)=h_0(t)\exp(x\cdot\beta)$. An event past the window `T_obs` is
right-censored (an empty sequence holding only the covariates and the window). We
hold out 30% of subjects for validation.

In [ ]:
BETA_TRUE = torch.tensor([0.7, -0.4])
seqs, meta = make_weibull_ph(
    n=1500, beta=BETA_TRUE, shape_k=2.0, scale_lambda=2.0, T_obs=3.0, seed=0
)
censoring = 1.0 - meta["observed"].float().mean()
print(f"subjects: {len(seqs)}    censoring rate: {censoring:.1%}")

# 70 / 30 train / validation split
g = torch.Generator().manual_seed(0)
perm = torch.randperm(len(seqs), generator=g).tolist()
cut = int(0.7 * len(seqs))
train_batch = survival_batch([seqs[i] for i in perm[:cut]])
val_batch = survival_batch([seqs[i] for i in perm[cut:]])

## 2. A density model over the event time

We put a **log-normal mixture** on the event time — the same distribution the
intensity-free model uses, imported straight from the library
(`LogNormalMixtureDistribution`). A `nn.Linear` maps the covariates $x$ to its
$3K$ parameters: $K$ locations, $K$ log-scales, $K$ log-weights.

`DensityBasedTPP` derives the whole contract from a few per-mark hooks. With one
event type ($D=1$) they are short. Signatures and shapes ($B$ subjects, $Q$ query
times):

- `_log_density_time_given_mark(t, ts)` → `(B, D)` or `(B, Q, D)`: $\log p(t\mid x)$,
  the log-density of the inter-event time $\tau = t - t_\text{last}$. With an empty
  history $t_\text{last}=0$, so $\tau$ is just the absolute time.
- `_log_survival_time_given_mark(t, ts)` → same shape: $\log S(\tau\mid x)$. Once a
  subject's single event has fired there is no next event, so we set $\log S = 0$
  for those rows — their censoring term must vanish.
- `_log_prob_mark(ts)` → `(B, D)`: $\log p(m\mid x)$. One event type, so $\log 1 = 0$.
- `_event_log_density_sum(ts)` → `(B,)`: the observed events' log-density, summed
  over the *scored* events (`scored_event_mask`).
- `_draw_next(ts, rng)`: draw one $(t, \text{type})$ from the mixture (used by `sample`).
- `time_mean(ts)` → `(B,)`: $E[t_\text{next}]$, closed-form for the mixture,
  $\sum_k w_k\exp(\mu_k + \tfrac12\sigma_k^2)$.

The base class assembles these into the likelihood
$\sum_i \log p(t_i\mid x) + \log S(T_\text{obs}\mid x)$: an observed subject
contributes its event-time density, a censored subject its survival. No hazard, no
time integral.

The hooks use two base-class helpers, `_last_event_time(ts)` (padding-aware last event time, 0 for an empty history) and `_get_termination_mask(ts)` (which types are still active). They are meant for subclasses but are internal and may change between versions.

In [ ]:
from torchpp.utils.event_data import scored_event_mask

# Numerical guards, mirroring the library's intensity-free model.
_EPS = 1e-8            # floor on the inter-event time (log-normal support is t > 0)
_LOG_SCALE_MIN = -2.0  # floor on the log-scale: a tiny scale gives huge log-density gradients
_LOG_SCALE_MAX = 3.0


class LogNormalMixtureSurvival(DensityBasedTPP):
    '''Survival model: a covariate-conditioned log-normal mixture on the event time.

    D=1 (one event type), terminating (fires at most once), has_origin (the event
    is the scored outcome).  `param_net` maps the covariates x to the 3K mixture
    parameters [K locations, K log-scales, K log-weights].
    '''

    def __init__(self, param_net, num_mix_components=8):
        super().__init__(D=1, terminating=True, token_roles=TokenRoles(has_origin=True))
        self.K = int(num_mix_components)
        self.param_net = param_net

    def _dist(self, raw):
        # Split the raw params into a log-normal mixture over the inter-event time.
        K = self.K
        locs = raw[..., :K]
        log_scales = raw[..., K:2 * K].clamp(_LOG_SCALE_MIN, _LOG_SCALE_MAX)
        log_weights = torch.log_softmax(raw[..., 2 * K:3 * K], dim=-1)
        return LogNormalMixtureDistribution(locs=locs, log_scales=log_scales, log_weights=log_weights)

    def _dist_and_tau(self, t, ts):
        # Per-subject mixture, and the inter-event time tau = t - t_last.
        raw = self.param_net(ts.covariates["x"])          # (B, 3K)
        t_last = self._last_event_time(ts)                # (B,) = 0 for an empty history
        if t.dim() == 2:                                  # (B, Q) query grid
            raw = raw.unsqueeze(1).expand(raw.shape[0], t.shape[1], raw.shape[-1])
            tau = (t - t_last.unsqueeze(1)).clamp(min=_EPS)
        else:
            tau = (t - t_last).clamp(min=_EPS)
        return self._dist(raw.to(t.dtype)), tau

    # --- density hooks ------------------------------------------------------
    def _log_density_time_given_mark(self, t, ts):
        if t.shape[0] == 0:
            return torch.zeros(*t.shape, self.D, dtype=t.dtype, device=t.device)
        dist, tau = self._dist_and_tau(t, ts)
        log_f = dist.log_prob(tau)                        # (B,) or (B, Q)
        return log_f.unsqueeze(-1).expand(*log_f.shape, self.D)   # broadcast over D=1

    def _log_survival_time_given_mark(self, t, ts):
        if t.shape[0] == 0:
            return torch.zeros(*t.shape, self.D, dtype=t.dtype, device=t.device)
        dist, tau = self._dist_and_tau(t, ts)
        log_S = dist.log_survival_function(tau)
        # A subject whose event has fired has no next event: log S = 0, so its
        # censoring term drops and the likelihood keeps only the event density.
        fired = ~self._get_termination_mask(ts).squeeze(-1)      # (B,)
        if log_S.dim() == 2:
            fired = fired.unsqueeze(1)
        log_S = log_S.masked_fill(fired, 0.0)
        return log_S.unsqueeze(-1).expand(*log_S.shape, self.D)

    def _log_prob_mark(self, ts):
        # One event type -> log p(m) = log 1 = 0.
        B = ts.time_points.shape[0]
        return torch.zeros(B, self.D, dtype=ts.time_points.dtype, device=ts.time_points.device)

    def _event_log_density_sum(self, ts):
        # Sum the observed events' log-density, over the scored events only.
        B, L = ts.time_points.shape
        dtype, device = ts.time_points.dtype, ts.time_points.device
        if L == 0:
            return torch.zeros(B, dtype=dtype, device=device)
        scored = scored_event_mask(ts, self.predictable_mask, self.has_origin)                    # (B, L)
        raw = self.param_net(ts.covariates["x"]).unsqueeze(1).expand(B, L, 3 * self.K)
        dist = self._dist(raw.to(dtype))
        prev = torch.cat([torch.zeros(B, 1, dtype=dtype, device=device), ts.time_points[:, :-1]], dim=1)
        tau = (ts.time_points - prev).clamp(min=_EPS)     # time since the previous event
        return dist.log_prob(tau).masked_fill(~scored, 0.0).sum(dim=1)

    # --- sampling + point prediction ---------------------------------------
    def _draw_next(self, ts_current, rng):
        batched = BatchedMVEventData.from_sequences([ts_current])   # keep the covariates
        dist = self._dist(self.param_net(batched.covariates["x"]))
        gmm = dist.base_dist
        k = torch.multinomial(torch.softmax(gmm.mixture_distribution.logits, -1), 1, generator=rng)
        loc = gmm.component_distribution.loc.gather(-1, k).squeeze(-1)
        scale = gmm.component_distribution.scale.gather(-1, k).squeeze(-1)
        tau = (loc + scale * torch.randn(loc.shape, generator=rng, device=loc.device)).exp()  # log-normal draw
        t_next = self._last_event_time(batched) + tau
        return t_next, torch.zeros(1, dtype=torch.long), torch.ones(1)

    def time_mean(self, ts):
        # E[t_next] = t_last + sum_k w_k exp(loc_k + 0.5 scale_k^2)  (log-normal mixture mean).
        K = self.K
        raw = self.param_net(ts.covariates["x"])
        locs = raw[..., :K]
        scales = raw[..., K:2 * K].clamp(_LOG_SCALE_MIN, _LOG_SCALE_MAX).exp()
        log_w = torch.log_softmax(raw[..., 2 * K:3 * K], dim=-1)
        e_tau = torch.logsumexp(log_w + locs + 0.5 * scales ** 2, dim=-1).exp()
        return self._last_event_time(ts) + e_tau

## 3. Fit by maximum likelihood

`fit_full_batch` (from `survival_tools`) runs full-batch Adam on the density negative log-likelihood — the shared
loop from the series, unchanged. Uncensored subjects contribute their event-time
density; censored subjects their survival.

In [ ]:
torch.manual_seed(0)
density_model = LogNormalMixtureSurvival(nn.Linear(2, 3 * 8), num_mix_components=8)
result = fit_full_batch(density_model, train_batch, steps=800, lr=0.02, val_batch=val_batch)
print(f"\nvalidation log-likelihood / event: {result['val_loglik']:.3f}")

## 4. Survival from the density

The survival curve reads straight off the density via `survival_time`.
Predictive curves use an empty-history batch from `covariate_batch` (see
[Predictive queries](./README.md#predictive-queries)).

To compare against the population **Kaplan-Meier** estimate (a non-parametric
survival curve over all subjects), we average the model's per-subject survival
$S(t\mid x_i)$ over subjects. We also check the identity $S(t) + \mathrm{CDF}(t) = 1$.

In [ ]:
t_grid = torch.linspace(0.02, 3.0, 60)

# per-subject survival S(t | x_i), averaged into one population curve
S_density = model_survival_curve(density_model, meta["x"], t_grid).mean(dim=0)

# non-parametric Kaplan-Meier over all subjects
t_km, S_km = kaplan_meier(meta["event_time"].clamp(max=3.0), meta["observed"])

fig = plot_survival_km_vs_model(t_km, S_km, t_grid, S_density.numpy(), labels=["density model"])
fig.axes[0].set_title("Density-model survival vs Kaplan-Meier")
plt.show()

## 5. Density route vs intensity route

Now fit an **intensity** model on the *same* data — the constant-hazard Cox model
from notebook 01 — and compare. On Weibull data its constant baseline hazard is
mis-specified, so it should fit worse. We compare held-out log-likelihood and both
survival curves against Kaplan-Meier.

In [ ]:
class ExponentialCoxTPP(IntensityBasedTPP):
    '''Constant-hazard proportional-hazards model (the intensity route, notebook 01).'''

    def __init__(self, n_features):
        super().__init__(D=1, terminating=True, token_roles=TokenRoles(has_origin=True))
        self.log_h0 = nn.Parameter(torch.zeros(()))       # log baseline hazard
        self.beta = nn.Parameter(torch.zeros(n_features))

    def _rate(self, ts):
        return torch.exp(self.log_h0) * torch.exp(ts.covariates["x"] @ self.beta)  # (B,)

    def _intensities_at_queries(self, query_times, ts):
        B, Q = query_times.shape
        return self._rate(ts).view(B, 1, 1).expand(B, Q, self.D)

    def _closed_form_ci(self, ts, T_low, T):
        hi = self._effective_T(T, ts)
        lo = self._effective_T(T_low, ts) if T_low is not None else 0.0
        return self._rate(ts).unsqueeze(-1) * (hi - lo)


torch.manual_seed(0)
cox_model = ExponentialCoxTPP(n_features=2)
cox_result = fit_full_batch(cox_model, train_batch, steps=800, lr=0.05, val_batch=val_batch, log_every=0)

print(f"density route   val log-lik / event: {result['val_loglik']:.3f}")
print(f"intensity route val log-lik / event: {cox_result['val_loglik']:.3f}")

S_cox = model_survival_curve(cox_model, meta["x"], t_grid).mean(dim=0)
fig = plot_survival_km_vs_model(t_km, S_km, t_grid, S_density.numpy(), labels=["density route"])
ax = fig.axes[0]
ax.plot(t_grid.numpy(), S_cox.numpy(), color="C3", linestyle=":", label="intensity route (constant hazard)")
ax.set_title("Both routes vs Kaplan-Meier")
ax.legend(fontsize=8)
plt.show()

## Takeaways

- **Two routes to one contract.** Model the hazard and integrate
  (`IntensityBasedTPP`), or model the event-time density directly
  (`DensityBasedTPP`). Both expose `likelihood`, `survival_time`, `sample`,
  `time_mean`, so they compare head-to-head.
- **A density survival model needs no encoder.** With one event per subject there
  is no history to embed; the density parameters are a plain map of the covariates.
  Six hooks define the model.
- **Flexibility pays off.** On Weibull data (rising hazard) the log-normal mixture
  tracks Kaplan-Meier closely, while the mis-specified constant-hazard model fits
  worse — in both survival shape and held-out log-likelihood.
- **Predictive queries use empty-history batches**; see [Predictive queries](./README.md#predictive-queries).
- The mixture's log-survival clamps the internal normal CDF to
  $[10^{-7},\, 1 - 10^{-7}]$ for numerical stability, so $S$ never reaches exactly
  0 or 1.